[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Validation and table=True


## What you will be able to do

Say which ways into a table model check a value and which do not, and show it: the constructor takes
anything, `model_validate` checks, an assignment does not until the class asks for it, and a row
loaded from the database is never checked at all. Put the checking at the edge of your program, where
data arrives, with a model that has no table. Recognize what a missing check looks like later: a
value that reaches a column it does not fit, a name longer than the column says, and a response model
that refuses what the table was happy to store.


## The idea

### The problem

A table model looks like a Pydantic model, and Pydantic models check their values. So a natural way
to write a service is to build a `Hero` from whatever arrived and let the model complain about
anything wrong with it. That is not what happens. The constructor of a table model checks nothing:
no type conversion, no length, no rule you wrote, nothing.

The result is a program that looks careful and is not. An age arrives as the word `old` and is
written into an `INTEGER` column, which SQLite accepts. A name arrives 80 characters long and goes
into a `VARCHAR(50)`, which SQLite also accepts. Neither is noticed at the line that made it. Both
are noticed much later, by something further away: a response model that refuses to serialize the
row, a report that adds up ages, or the day the same code runs against PostgreSQL, which refuses
what SQLite allowed.

### Which ways in check a value

> A **table model's constructor does not validate**. `Hero(age="old")` builds the object and writes
> `"old"` where an `int` was declared. **`Hero.model_validate(data)`** does validate: it converts,
> applies every `field_validator` and refuses what it cannot make fit. An **assignment**,
> `hero.age = "old"`, does not validate either. A class that sets
> **`model_config = {"validate_assignment": True}`** checks both of those, since SQLModel's
> constructor assigns each value in turn. A row **loaded from the database** is never validated,
> whatever the class says. A model **without `table=True`** validates in its constructor,
> like any Pydantic model, which is what makes it the right place to check what arrives.

### Why it works that way

- **The class has two jobs, and loading is one of them.** Every row read becomes an instance of the
  same class, and checking each one against today's rules would refuse rows that were written
  before those rules existed, for a value the program is only trying to read.
- **The constructor is what loading uses.** That is why it lets everything through, and why the
  check has to be somewhere you call on purpose.
- **`model_validate` is that place.** It is one call, it converts as well as refuses, and it is what
  a route or a loader should use on anything from outside.
- **A model with no table is the cleaner place still.** It validates in its constructor, it can
  leave out the fields a client may not set, and FastAPI calls it for you, which the
  **SQLModel in FastAPI** notebook shows.
- **The database is the last check, and it is not the same check.** It knows `NOT NULL`, uniqueness
  and foreign keys. It does not know that a name starts with a capital letter, and SQLite does not
  even enforce the length of a `VARCHAR`.

### Where this shows up

Anywhere data comes from outside the program: a request body, a CSV, a message, another service.
The **APIs and JSON** guide's Schemas and Validation notebook is where Pydantic's checking is taught
in full, including `field_validator`; this notebook is about which of SQLModel's doors run it. The
**Create, Read and Update Models** notebook builds the family of models that puts the checking where
it belongs.

### What this notebook covers

- The same class, with a table and without
- The four ways in, and which of them check
- What the database checks, and what SQLite lets through
- Checking at the edge, with a model that has no table
- Turning assignment checking on
- A hero accepted at the edge, finished
- Three failures, all of them found a long way from where they were made

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlmodel import Field, SQLModel


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    age: int | None = None


class HeroDraft(SQLModel):                      # the same fields, no table
    name: str
    age: int | None = None


print("table model  :", repr(Hero(name="Deadpond", age="old").age))
try:
    HeroDraft(name="Deadpond", age="old")
except Exception as error:
    reason = str(error).splitlines()[2].strip().split(",")[0]
    print("plain model  :", type(error).__name__, "|", reason)
```

```
table model  : 'old'
plain model  : ValidationError | Input should be a valid integer
```

Two classes with the same fields, and only one of them refuses the word `old` where a number was
declared. The one that refuses is the one with no table. The one that accepts is the one that writes
rows, which is the whole of this notebook's subject.


## Setup

Ten imports, one of them installed first where it is missing, the cast, two helpers, the classes,
the engine, and the database built and loaded.

- `sqlmodel` is the library, and `SQLModel`, `Field`, `Session`, `create_engine` and `select`, from
  it, are the classes, the session and the reading. Colab does not have SQLModel, so the cell
  installs 0.0.42 with `pip` where it is missing, and `version` and `PackageNotFoundError`, from
  `importlib.metadata`, `subprocess` and `sys` find out whether it is
- `ValidationError` and `field_validator`, from `pydantic`, are the error every check raises and the
  decorator that writes a rule of your own
- `event`, `insert` and `text`, from `sqlalchemy`, are the pragma on every connection, the rows
  loaded without a session, and the one row this notebook writes as plain SQL
- `re` takes memory addresses out of a message, `Path` names the database file, and `shutil` removes
  the scratch folder at the start and at the end
- `TEAMS` and `HEROES` are the cast, which `build` loads

The classes, `hero_engine` and `build` are the **Engine and create_all** and **Sessions** notebooks'.
This notebook adds two more of its own, in the first worked example.


In [1]:
import re
import shutil
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from pydantic import ValidationError, field_validator
from sqlalchemy import event, insert, text
from sqlmodel import Field, Session, SQLModel, create_engine, select

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


def build(engine):
    """Create the tables and load the cast, without a session: every notebook starts from the same rows."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": where} for name, where in TEAMS])
        teams = {name: number for number, (name, _) in enumerate(TEAMS, start=1)}
        connection.execute(insert(Hero), [{"name": name, "secret_name": secret, "age": age,
                                           "team_id": teams.get(team)}
                                          for name, secret, age, team in HEROES])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same eight heroes
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes loaded")


sqlmodel 0.0.42 | 8 heroes loaded


## Worked examples

### The same class, with a table and without

Two classes, the same two fields, and the same rule written on both. One of them has a table:


In [2]:
class CheckedHero(SQLModel, table=True):
    """The same fields as Hero, with a rule about names, and a table."""

    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    age: int | None = None

    @field_validator("name")
    @classmethod
    def starts_with_a_capital(cls, value):
        print("      (the rule ran)")
        if not value[:1].isupper():
            raise ValueError("a hero's name starts with a capital letter")
        return value


class HeroDraft(SQLModel):
    """The same fields and the same rule, with no table."""

    name: str = Field(max_length=50)
    age: int | None = None

    @field_validator("name")
    @classmethod
    def starts_with_a_capital(cls, value):
        print("      (the rule ran)")
        if not value[:1].isupper():
            raise ValueError("a hero's name starts with a capital letter")
        return value


SQLModel.metadata.create_all(engine)
print("the table model:")
kept = CheckedHero(name="deadpond", age="old")
print("     name:", repr(kept.name), "| age:", repr(kept.age))

print("the model with no table:")
try:
    HeroDraft(name="deadpond", age="old")
except ValidationError as error:
    print(message(error))


the table model:
     name: 'deadpond' | age: 'old'
the model with no table:
      (the rule ran)
2 validation errors for HeroDraft
name
  Value error, a hero's name starts with a capital letter [type=value_error, input_value='deadpond', input_type=str]
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='old', input_type=str]


The rule printed a line whenever it ran, and it did not run for the table model: no line, a lowercase
name kept, and the word `old` sitting in a field declared `int | None`. On the model with no table
the rule ran, refused the lowercase name, and the age was refused as well, so the message names both
problems at once, which is what Pydantic hands back for a body with more than one thing wrong.

The difference is `table=True` and nothing else. The two classes were written from the same lines.

### The four ways in, and which of them check

A value can reach a table model four ways. Here they are in order, with what each one does:


In [3]:
print("1. the constructor")
loose = CheckedHero(name="deadpond", age="old")
print("     kept:", repr(loose.name), repr(loose.age))

print("2. model_validate")
try:
    CheckedHero.model_validate({"name": "deadpond", "age": "48"})
except ValidationError as error:
    print("     ValidationError:", message(error).splitlines()[-1].strip()[:60])
print("     a good one:", repr(CheckedHero.model_validate({"name": "Deadpond", "age": "48"}).age))

print("3. an assignment")
loose.age = "older still"
print("     kept:", repr(loose.age))

print("4. a row loaded from the database")
with engine.begin() as connection:
    connection.execute(text("INSERT INTO checkedhero (name, age) VALUES ('deadpond', 'old')"))
with Session(engine) as session:
    loaded = session.exec(select(CheckedHero)).one()
    print("     read back:", repr(loaded.name), repr(loaded.age))


1. the constructor
     kept: 'deadpond' 'old'
2. model_validate
      (the rule ran)
     ValidationError: Value error, a hero's name starts with a capital letter [typ
      (the rule ran)
     a good one: 48
3. an assignment
     kept: 'older still'
4. a row loaded from the database
     read back: 'deadpond' 'old'


| The way in | Converts | Runs your rules | Notes |
|---|---|---|---|
| `CheckedHero(...)` | no | no | anything at all, kept as it was given |
| `CheckedHero.model_validate(...)` | yes | yes | `"48"` became `48`, and the lowercase name was refused |
| `hero.age = ...` | no | no | both of these change with `validate_assignment`, below |
| a row read from the table | no | no | whatever the column happens to hold |

Only the second checks anything, and it is the only one a program calls on purpose. The fourth is the
reason for the other three: every row read from the table is built through the same constructor, so a
constructor that validated would refuse rows that were written before a rule existed, or by another
program, or by hand.

### What the database checks, and what SQLite lets through

The database is the last line, and it checks fewer things than people expect:


In [4]:
with Session(engine) as session:
    session.add(CheckedHero(name="x" * 80, age="old"))              # 80 characters into a VARCHAR(50)
    session.commit()

with Session(engine) as session:
    longest = max(session.exec(select(CheckedHero)).all(), key=lambda hero: len(hero.name))
    print("name length stored:", len(longest.name), "| the column says:",
          CheckedHero.__table__.columns["name"].type.length)
    print("age stored        :", repr(longest.age), type(longest.age).__name__)


name length stored: 80 | the column says: 50
age stored        : 'old' str


Eighty characters went into a column declared `VARCHAR(50)`, and the word `old` went into one
declared `INTEGER`. Neither is a bug in SQLModel: SQLite does not enforce the length of a `VARCHAR`,
and its columns have a preferred type rather than a required one, so a string in an `INTEGER` column
is stored as a string. PostgreSQL, MySQL and SQL Server all refuse both of those, which means the
same code that passes its tests on SQLite can fail on the server it is deployed to. What every
database does enforce is `NOT NULL`, uniqueness and foreign keys, which the **Sessions** notebook
showed refusing rows.

### Checking at the edge, with a model that has no table

The answer is not to distrust table models. It is to check where data arrives, with a model whose
constructor checks, and to build the table model from what came through:


In [5]:
def accept(raw):
    """Check what arrived, and turn it into a row, or say what was wrong with it."""
    try:
        draft = HeroDraft.model_validate(raw)                       # the rules run here
    except ValidationError as error:
        return None, message(error).splitlines()[-1].strip()[:60]
    return CheckedHero.model_validate(draft.model_dump()), None


for raw in [{"name": "Tarantula", "age": "32"}, {"name": "deadpond", "age": 30}, {"name": "Ghost", "age": "old"}]:
    hero, refused = accept(raw)
    print(f"  {str(raw):<44} -> {fields(hero) if hero else refused}")


      (the rule ran)
      (the rule ran)
  {'name': 'Tarantula', 'age': '32'}           -> {'id': None, 'name': 'Tarantula', 'age': 32}
      (the rule ran)
  {'name': 'deadpond', 'age': 30}              -> Value error, a hero's name starts with a capital letter [typ
      (the rule ran)
  {'name': 'Ghost', 'age': 'old'}              -> Input should be a valid integer, unable to parse string as a


One row accepted and converted, two refused with the reason, and nothing written until the values are
known to be what the class says they are. `HeroDraft` has no `id`, which is right for something a
client sends, and no table, so it exists only for as long as the check takes.

### One line that makes a table model check

A table model can be told to check assignments, and that closes the constructor as well, because
SQLModel builds an object by assigning each value in turn:


In [6]:
class StrictHero(SQLModel, table=True):
    model_config = {"validate_assignment": True}

    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    age: int | None = None


SQLModel.metadata.create_all(engine)
try:
    StrictHero(name="Tarantula", age="not a number")
except ValidationError as error:
    print("the constructor:", message(error).splitlines()[-1].strip()[:60])

strict = StrictHero(name="Tarantula", age="32")                     # and it converts, as model_validate does
print("converted      :", repr(strict.age), type(strict.age).__name__)
try:
    strict.age = "still not a number"
except ValidationError as error:
    print("an assignment  :", message(error).splitlines()[-1].strip()[:60])

with engine.begin() as connection:                                  # a row written past the model
    connection.execute(text(f"INSERT INTO {StrictHero.__tablename__} (name, age) VALUES ('loose', 'old')"))
with Session(engine) as session:
    loose_row = session.exec(select(StrictHero)).one()
    print("a row loaded   :", repr(loose_row.name), repr(loose_row.age))


the constructor: Input should be a valid integer, unable to parse string as a
converted      : 32 int
an assignment  : Input should be a valid integer, unable to parse string as a
a row loaded   : 'loose' 'old'


Three of the four doors are now closed, with one line. The constructor refuses what it cannot
convert, converts what it can, and an assignment does the same. The fourth is unchanged: the row
written past the model came back with a lowercase name and the word `old` in an `int` field, because
loading does not go through either door.

That makes it a real option, and it is not free. Every assignment is validated, including the ones a
program makes in a loop, and a rule that reads other fields is run before the object is finished
being built. It also says nothing about the rows already in the table. Where the data comes from
outside, a model with no table is still the clearer place to check, because it can refuse fields a
client may not set at all.

### A hero accepted at the edge, finished

The pieces of this notebook in one function. `add_hero` is what a route would call: check at the
edge, build the row from what passed, write it, and give back only the fields a response should
carry:


In [7]:
class HeroSummary(SQLModel):                                        # what goes back out
    id: int
    name: str


def add_hero(engine, raw):
    """Check, write, and return the summary, or the reason the data was refused."""
    try:
        draft = HeroDraft.model_validate(raw)
    except ValidationError as error:
        return {"refused": message(error).splitlines()[-1].strip()[:56]}
    hero = CheckedHero.model_validate(draft.model_dump())
    with Session(engine, expire_on_commit=False) as session:
        session.add(hero)
        session.commit()
    return HeroSummary.model_validate(hero).model_dump()


for raw in [{"name": "Black Lion", "age": "35"}, {"name": "lowercase lion", "age": 35},
            {"name": "Dr. Weird", "age": "ancient"}]:
    print(" ", add_hero(engine, raw))


      (the rule ran)
      (the rule ran)
  {'id': 3, 'name': 'Black Lion'}
      (the rule ran)
  {'refused': "Value error, a hero's name starts with a capital letter "}
      (the rule ran)
  {'refused': 'Input should be a valid integer, unable to parse string '}


One hero written and summarized, and two refused before any database was asked, each with the reason
the check gave. The summary carries the id and the name and nothing else, which is a model with no
table doing the job it is best at. The **Create, Read and Update Models** notebook makes a family of
these and gives them a shared base, so the fields are written once.

### Where each part came from

| In `add_hero` | What it relies on | The section that showed it |
|---|---|---|
| `HeroDraft.model_validate(raw)` | a model with no table, whose rules run | The same class, with a table and without |
| `CheckedHero.model_validate(...)` | the one door into a table model that checks | The four ways in, and which of them check |
| `expire_on_commit=False` | an object still readable after its session closed | the **Sessions** notebook |
| `HeroSummary.model_validate(hero)` | a model with no table, reading the fields off an object | Checking at the edge |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/06-validation-and-table-true-solutions.ipynb).

**1.** Build a `CheckedHero` with an age of `"twelve"` and print the value and its type. Then do the
same through `model_validate` and print what it says.


In [8]:
# your code here


**2.** Write `TeamDraft`, a model with no table, with a `name` of at most 50 characters and a
`headquarters` of at most 60, and a rule that refuses a headquarters of fewer than 3 characters. Show
it refusing one and accepting another.


In [9]:
# your code here


**3.** Write a function that takes a list of raw dictionaries and returns two lists, the drafts that
passed and the reasons the others were refused.


In [10]:
# your code here


**4.** Write a `StrictTeam` table model with `validate_assignment` on, and show what its constructor
and an assignment each do with a year that is not a number.


In [11]:
# your code here


**5.** Write a row into `team` with plain SQL whose `name` is 80 characters long, read it back with
the model, and print the length of what came back beside the length the column declares.


In [12]:
# your code here


**6.** Write `TeamSummary`, a model with no table carrying the id and the name, and turn a `Team`
read from the database into one, printing what it holds.


In [13]:
# your code here


## Common errors

### No error, and a rule that never ran: a value given to the constructor


In [14]:
straight = CheckedHero(name="rusty-man", age="quite old")           # both are wrong, and nothing says so
print("what the object holds:", fields(straight))
with Session(engine) as session:
    session.add(straight)
    session.commit()
print("and it is in the table")


what the object holds: {'id': None, 'name': 'rusty-man', 'age': 'quite old'}
and it is in the table


The rule about capital letters is on the class, and it did not run. The type on the field is
`int | None`, and the word `quite old` is in it. The row is written, and the only thing that went
wrong happened in a constructor that raised nothing.

This is the failure the rest of this notebook exists to prevent, and the fix is one call:


In [15]:
try:
    CheckedHero.model_validate({"name": "rusty-man", "age": "quite old"})
except ValidationError as error:
    print(message(error))


      (the rule ran)
2 validation errors for CheckedHero
name
  Value error, a hero's name starts with a capital letter [type=value_error, input_value='rusty-man', input_type=str]
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='quite old', input_type=str]


### No error, and a value assigned after it was checked


In [16]:
good = CheckedHero.model_validate({"name": "Tarantula", "age": "32"})
print("checked on the way in:", repr(good.age))
good.age = "not a number any more"
print("and after an assignment:", repr(good.age))


      (the rule ran)
checked on the way in: 32
and after an assignment: 'not a number any more'


The object was built through the one door that checks, and then an ordinary attribute assignment put
anything at all in it. In a service this is the line in the middle of a route that updates a field
from a request, long after whatever validated the body.

`model_config = {"validate_assignment": True}` on the class is one fix, as `StrictHero` showed
above, and the **Create, Read and Update Models** notebook has the other: validate the incoming
change into a model of its own, then copy it over.

### ValidationError: 1 validation error for HeroReport


In [17]:
class HeroReport(SQLModel):                                         # what a response would carry
    id: int
    name: str
    age: int                                                        # a number, and required


with Session(engine) as session:
    stored = session.exec(select(CheckedHero).where(CheckedHero.age == "old")).first()
    print("the row says:", fields(stored))
    HeroReport.model_validate(stored)


the row says: {'id': 1, 'name': 'deadpond', 'age': 'old'}


ValidationError: 1 validation error for HeroReport
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='old', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing

This is where an unchecked value finally shows: not in the constructor that accepted it, not in the
commit that wrote it, but in the model that tries to send it out, hours or months later, in a route
that has nothing wrong with it. The message names the field and what it expected, and that is all the
information there is: nothing says which line wrote the row.

The answer is upstream, and there are two halves to it. Check what arrives, so a bad value is never
written, and give the column a shape the database itself will enforce, which is what a
`CHECK` constraint does and what the **sa_column and __table_args__** notebook covers.

Last, the engine lets go of the file, and this cell removes the scratch folder with the database in
it:


In [18]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A table model's constructor validates nothing: no conversion, no length, no rule of your own, and
  an assignment does no better.
- `Model.model_validate(data)` is the one door in that checks, and it converts as well as refuses.
- A row read from the table is never validated, which is why the constructor cannot be.
- A model with no `table=True` validates in its constructor, which makes it the right place to check
  what arrives and the right shape for what goes out.
- SQLite enforces `NOT NULL`, uniqueness and foreign keys, and not a `VARCHAR` length or a column's
  type, so a value that passes here can fail on another database.


## What is next

The **Create, Read and Update Models** notebook turns those loose drafts and summaries into a family:
a shared base, a create model, a public model and an update model, `model_validate` between them, and
`sqlmodel_update` with `exclude_unset`, which is what stops a partial update writing `None` over
every field the caller did not mention.


---

&#8592; **Previous:** [Reading Rows](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/05-reading-rows.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
